# Splatastrophe: Barn and PhotoTourism reconstruction
Run on Thunder/VS Code or Colab with a separate reconstruction environment. Select one scene below. Commands preview until EXECUTE=True. GPU installation/training must be validated on the selected machine; no reconstruction results are bundled.


In [ ]:
from pathlib import Path
import platform
import sys
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())


## Locate or clone the repository
In VS Code, run the notebook from the cloned repository. In Colab, add a secret named `GITHUB_TOKEN` with read access to this repository, enable notebook access, then run the next cell. The token is supplied through a temporary askpass helper and is not stored in the remote URL. The checkout defaults to `/content/Splatastrophe`. A shared notebook does not share its runtime or credentials.


In [ ]:
import os
import subprocess
import tempfile

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()
if IS_COLAB:
    REPO = Path("/content/Splatastrophe")
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").is_file()), None)
    if REPO is None:
        raise RuntimeError("Open this notebook from the Splatastrophe checkout.")

if not (REPO / ".git").exists():
    if not IS_COLAB:
        raise RuntimeError("Clone the repository first.")
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add the GITHUB_TOKEN secret with repository read access.")
    with tempfile.TemporaryDirectory() as temp_dir:
        helper = Path(temp_dir) / "askpass.py"
        helper.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["SPLAT_GITHUB_TOKEN"])\n')
        helper.chmod(0o700)
        child_env = dict(os.environ, GIT_ASKPASS=str(helper), GIT_TERMINAL_PROMPT="0", SPLAT_GITHUB_TOKEN=token)
        try:
            subprocess.run(["git", "-c", "credential.helper=", "clone", "--branch", "sriya/reconstruction", "https://github.com/carldjapardi/Splatastrophe.git", str(REPO)], env=child_env, check=True)
        finally:
            child_env.pop("SPLAT_GITHUB_TOKEN", None)
            del token
print("Repository:", REPO)


In [ ]:
subprocess.run([sys.executable, str(REPO / "scripts/check_runtime.py"), "--output", str(REPO / "outputs/runtime.json")], check=True)


## Experiment outputs
Use a unique run name. Large outputs are ignored by Git. Colab runtime storage is temporary: copy checkpoints and exports to persistent storage before ending a session. On Thunder, verify instance/storage lifecycle and save artifacts before stopping or deleting compute.


In [ ]:
from datetime import datetime, timezone
RUN_NAME = "runtime-check-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO / "outputs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", RUN_DIR)


## Choose one scene and storage root
Run Barn through export first, then repeat for PhotoTourism Brandenburg Gate. The official PhotoTourism archive is about 4 GB. Keep source data and outputs on persistent disk. Download preserves originals and records archive/split hashes. Barn estimates cameras; PhotoTourism reuses supplied COLMAP intrinsics and poses.


In [ ]:
EXECUTE = False  # Enable only on the selected GPU environment.
SCENE = "barn"  # or "brandenburg-gate" for PhotoTourism
DATA_ROOT = REPO  # Set to a mounted persistent directory if available.
HOURLY_RATE_USD = None  # Set to the displayed rate; stage estimate excludes setup/idle time.
def stage(name, *extra):
    command = [sys.executable, str(REPO / "pipeline/reconstruction/run.py"), name,
               "--scene", str(REPO / f"configs/scenes/{SCENE}.json"), "--data-root", str(DATA_ROOT), *extra]
    if HOURLY_RATE_USD is not None:
        command += ["--hourly-rate-usd", str(HOURLY_RATE_USD)]
    if EXECUTE:
        command.append("--execute")
    subprocess.run(command, cwd=REPO, check=True)


In [ ]:
stage("download")  # Once per scene; refuses to overwrite an existing raw directory.
stage("process")


In [ ]:
stage("split")
stage("inspect")


In [ ]:
# Inspect data/processed/<scene>/alignment.html and COLMAP sparse geometry first.
stage("train")  # 1,000-iteration smoke test; do not start a longer job before export validation.


## Evaluate, export, and preview
Use the exact config printed by a successful training run. Train/validation/test are disjoint. NeRF-W test membership is preserved for PhotoTourism; full-image Splatfacto scores do not reproduce NeRF-W appearance-adapted benchmark scores. Input camera metadata and dataparser transforms are packaged with building.ply. Browser integration remains a separate verification.


In [ ]:
TRAIN_CONFIG = ""  # e.g. outputs/barn/barn/splatfacto/<timestamp>/config.yml
if TRAIN_CONFIG:
    config = Path(TRAIN_CONFIG)
    if not config.is_absolute():
        config = DATA_ROOT / config
    stage("eval", "--load-config", str(config))
    stage("export", "--load-config", str(config))
    stage("preview", "--load-config", str(config))
else:
    print("Set TRAIN_CONFIG after a successful GPU training run.")
